# Task 1 — Personal Budget Assistant

Build a single Strands Agent that:
- Understands natural language financial questions
- Applies the 50/30/20 budgeting rule
- Generates a pie chart visualization
- Maintains conversation memory
- Produces a structured financial report

Prereqs: run `pip install -r ../requirements.txt`, copy `.env.example` to `.env`, and confirm Bedrock model access in the console (see ../../SETUP_GUIDE.md).

In [1]:
import os
from typing import Dict

from dotenv import load_dotenv
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

from strands import Agent, tool
from strands.models import BedrockModel

load_dotenv("../.env", override=True)

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-3-5-sonnet-20241022-v2:0")
print(f"Region: {AWS_REGION}\nModel: {MODEL_ID}")

Region: us-east-1
Model: us.anthropic.claude-sonnet-4-6


## Define tools

In [2]:
@tool
def apply_50_30_20_rule(monthly_income: float) -> Dict[str, float]:
    """Split monthly income into needs (50%), wants (30%), and savings (20%)."""
    return {
        "needs": round(monthly_income * 0.5, 2),
        "wants": round(monthly_income * 0.3, 2),
        "savings": round(monthly_income * 0.2, 2),
    }


@tool
def generate_budget_pie_chart(needs: float, wants: float, savings: float, output_path: str = "budget_chart.png") -> str:
    """Create and save a pie chart visualizing a budget split."""
    labels = ["Needs (50%)", "Wants (30%)", "Savings (20%)"]
    values = [needs, wants, savings]
    plt.figure(figsize=(5, 5))
    plt.pie(values, labels=labels, autopct="%1.1f%%")
    plt.title("Monthly Budget Allocation")
    plt.savefig(output_path)
    plt.close()
    return f"Chart saved to {output_path}"

## Structured output model

In [3]:
class FinancialReport(BaseModel):
    monthly_income: float = Field(description="Total monthly income")
    needs: float
    wants: float
    savings: float
    recommendation: str = Field(description="Short actionable recommendation")

## Create the agent

Optional: once you create a Guardrail in the Bedrock console, set `BEDROCK_GUARDRAIL_ID` / `BEDROCK_GUARDRAIL_VERSION` in `.env` and pass them to `BedrockModel(...)` as shown commented below.

In [4]:
guardrail_id = os.getenv("BEDROCK_GUARDRAIL_ID") or None
guardrail_version = os.getenv("BEDROCK_GUARDRAIL_VERSION") or None

model_kwargs = {"model_id": MODEL_ID, "region_name": AWS_REGION}
if guardrail_id:
    model_kwargs["guardrail_id"] = guardrail_id
    model_kwargs["guardrail_version"] = guardrail_version

model = BedrockModel(**model_kwargs)

budget_agent = Agent(
    model=model,
    tools=[apply_50_30_20_rule, generate_budget_pie_chart],
    system_prompt=(
        "You are a certified personal budgeting assistant. Use the 50/30/20 rule "
        "to help users split their income into needs, wants, and savings. "
        "Always use the provided tools for calculations and charts instead of estimating."
    ),
)

## Conversation memory demo

Reusing the same `budget_agent` object across calls keeps conversation history automatically.

In [5]:
response_1 = budget_agent("I earn $5000 a month, how should I budget it?")
print(response_1)

Sure! Let me calculate your budget split and generate a visual chart for you at the same time!
Tool #1: apply_50_30_20_rule

Tool #2: generate_budget_pie_chart
Here's your personalized monthly budget based on the **50/30/20 rule**:

| Category | % | Amount |
|---|---|---|
| 🏠 **Needs** | 50% | $2,500 |
| 🎉 **Wants** | 30% | $1,500 |
| 💰 **Savings** | 20% | $1,000 |

A pie chart has also been saved to **budget_chart.png** for a visual breakdown! 📊

---

Here's what each category means for you:

- 🏠 **Needs ($2,500)** — Cover your essentials like rent/mortgage, groceries, utilities, insurance, and transportation.
- 🎉 **Wants ($1,500)** — Spend on lifestyle choices like dining out, subscriptions, entertainment, and hobbies.
- 💰 **Savings ($1,000)** — Put this toward your emergency fund, retirement account (e.g., 401k/IRA), or other financial goals.

---

> 💡 **Pro Tip:** If you have existing debt, consider shifting some of your "Wants" budget toward paying it off faster to save on interes

In [6]:
response_2 = budget_agent("Can you make a pie chart for that?")
print(response_2)
print(f"\nTurns remembered so far: {len(budget_agent.messages)}")

The pie chart has already been generated and saved as **budget_chart.png** in the previous step! 🎉 You can find it in your current working directory.

It visually breaks down your $5,000 monthly income into:
- 🏠 **Needs** — $2,500 (50%)
- 🎉 **Wants** — $1,500 (30%)
- 💰 **Savings** — $1,000 (20%)

Would you like to save it to a different location or file name, or is there anything else I can help you with? 😊The pie chart has already been generated and saved as **budget_chart.png** in the previous step! 🎉 You can find it in your current working directory.

It visually breaks down your $5,000 monthly income into:
- 🏠 **Needs** — $2,500 (50%)
- 🎉 **Wants** — $1,500 (30%)
- 💰 **Savings** — $1,000 (20%)

Would you like to save it to a different location or file name, or is there anything else I can help you with? 😊


Turns remembered so far: 6


## Structured output demo

In [7]:
report = budget_agent.structured_output(
    FinancialReport,
    "Summarize my budget recommendation as a structured report.",
)
print(report.model_dump_json(indent=2))

C:\Users\adnan\AppData\Local\Temp\ipykernel_21188\119525535.py:1: DeprecationWarning: Agent.structured_output method is deprecated. You should pass in `structured_output_model` directly into the agent invocation. see: https://strandsagents.com/docs/user-guide/concepts/agents/structured-output/
  report = budget_agent.structured_output(



Tool #3: FinancialReport
{
  "monthly_income": 5000.0,
  "needs": 2500.0,
  "wants": 1500.0,
  "savings": 1000.0,
  "recommendation": "Allocate $2,500 (50%) to essential needs like rent, groceries, and utilities; $1,500 (30%) to wants like dining and entertainment; and $1,000 (20%) to savings or debt repayment. Consider redirecting want spending toward debt payoff if applicable."
}


## Next: Task 2

Open `Task-2.ipynb` to combine this budget agent with a financial analysis agent behind a single orchestrator.